# Technique 2: Case 3

Logarithmic smoothing; q penalty A=10; factor=0.1; no exact residual cache. At most 120 Taylor steps per propagation.

Run all cells to repeat all methods. Convergence and serial %%timeit use separate cells; timing uses 500 initial guesses and seven repeats.


In [1]:
import os
import sys
import copy
import importlib
import json
import platform
from pathlib import Path

for name in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS',
             'VECLIB_MAXIMUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    os.environ[name] = '1'

import numpy as np
import scipy
import heyoka as hy
from IPython.display import display, Markdown

hy.set_nthreads(1)


In [2]:
# =========================
# Timing Settings
# =========================
from collections import Counter

cpu_samples = []
fast_cpus = []
if sys.platform == 'darwin':
    import ctypes
    import plistlib
    import subprocess
    tree = plistlib.loads(subprocess.check_output(
        ['/usr/sbin/ioreg', '-a', '-l', '-p', 'IODeviceTree', '-n', 'cpus', '-r']))
    pending = list(tree)
    while pending:
        node = pending.pop()
        if node.get('cluster-type') == b'P\x00' and 'cpu-id' in node:
            fast_cpus.append(int.from_bytes(node['cpu-id'], 'little'))
        pending.extend(node.get('IORegistryEntryChildren', []))
    lib = ctypes.CDLL('/usr/lib/libSystem.B.dylib')
    lib.pthread_set_qos_class_self_np.argtypes = [ctypes.c_uint, ctypes.c_int]
    lib.pthread_cpu_number_np.argtypes = [ctypes.POINTER(ctypes.c_size_t)]

def set_timing_priority():
    if sys.platform == 'darwin':
        assert lib.pthread_set_qos_class_self_np(0x21, 0) == 0

def sample_cpu():
    if sys.platform == 'darwin':
        cpu = ctypes.c_size_t()
        assert lib.pthread_cpu_number_np(ctypes.byref(cpu)) == 0
        cpu_samples.append(cpu.value)

environment = dict(python=sys.version, executable=sys.executable,
                   platform=platform.platform(), machine=platform.machine(),
                   numpy=np.__version__, scipy=scipy.__version__, heyoka=hy.__version__,
                   threads=1, fast_cpus=sorted(fast_cpus), strict_cpu_pinning=False)
print(environment)

records_by_method = {}
results_by_method = {}

def save_trial(i, result, accepted, nfev):
    records.append(dict(trial=int(i), accepted=bool(accepted), nfev=int(nfev),
        status=int(result.status) if result is not None else None,
        residual_norm=float(np.linalg.norm(result.fun)) if result is not None else None,
        x=result.x.tolist() if result is not None else None))


{'python': '3.12.14 (main, Sep  3 2026, 03:46:19) [Clang 21.1.8 ]', 'executable': '/Users/zhong/Documents/Codex/miniforge3/envs/gtoc13_heyoka7121_paper/bin/python', 'platform': 'macOS-26.5.1-arm64-arm-64bit', 'machine': 'arm64', 'numpy': '2.5.3', 'scipy': '1.18.0', 'heyoka': '7.12.1', 'threads': 1, 'fast_cpus': [12, 13, 14, 15, 16, 17], 'strict_cpu_pinning': False}


In [3]:
import numpy as np

# [modify-lunar] Use the Cartesian RV+m dynamics with strict logarithmic-reduced Hamiltonian.
# The module is the modified dynamics file produced for the lunar case.

# =========================
# Setup (SI Units)
# =========================
# [modify-lunar] Lunar model and propulsion parameters agreed for Case 3.
mu = 4.902800118e12       # lunar gravitational parameter [m^3/s^2]
R_M = 1.738000e6          # lunar reference radius [m]
m0 = 874.4                # initial mass [kg]
Tmax = 2200.0             # maximum thrust [N]
c = 3090.2                # effective exhaust velocity [m/s]
u_min = 0.4               # minimum throttle [-]
u_max = 1.0               # maximum throttle [-]

# [modify-lunar] Initial and terminal Cartesian states in the Moon-centered inertial frame.
r0 = np.array([1.753000e6, 0.0, 0.0], dtype=float)       # [m]
v0 = np.array([400.0, 800.0, 1100.0], dtype=float)       # [m/s]
rf = np.array([1.66937673e6, 3.3845930e5, 3.4535590e5], dtype=float)  # [m]
vf = np.array([0.0, 0.0, 0.0], dtype=float)              # [m/s]

t0 = 0.0                # initial time [s]
# [modify-lunar] tf is free and is solved together with the initial costates.

# =========================
# Nondimensionalize
# =========================
# [modify-lunar] Characteristic units: 1 LU = R_M, 1 TU = sqrt(R_M^3/mu), 1 MU = m0.
LUnit = R_M
TUnit = np.sqrt(LUnit**3 / mu)
MUnit = m0

VUnit = LUnit / TUnit
AUnit = VUnit / TUnit
FUnit = MUnit * AUnit

# [modify-lunar] Dimensionless state and physical parameters.
x0_nu = np.concatenate((r0 / LUnit, v0 / VUnit)).tolist()
xf_nu = np.concatenate((rf / LUnit, vf / VUnit)).tolist()
t0_nu = t0 / TUnit
m0_nu = m0 / MUnit
mu_nu = mu * TUnit**2 / LUnit**3
Tmax_nu = Tmax / FUnit
c_nu = c / VUnit

# Preserve the original short parameter names used by the shooting code.
c1 = Tmax_nu
c2 = c_nu

print(f"LUnit = {LUnit:.9e} m")
print(f"TUnit = {TUnit:.9e} s")
print(f"VUnit = {VUnit:.9e} m/s")
print(f"FUnit = {FUnit:.9e} N")
print(f"mu_nu = {mu_nu:.16e}")
print(f"Tmax_nu = {Tmax_nu:.16e}")
print(f"c_nu = {c_nu:.16e}")
print("x0_nu =", np.asarray(x0_nu))
print("xf_nu =", np.asarray(xf_nu))
print(f"||rf|| / R_M = {np.linalg.norm(rf) / R_M:.16e}")


LUnit = 1.738000000e+06 m
TUnit = 1.034790813e+03 s
VUnit = 1.679566516e+03 m/s
FUnit = 1.419236568e+03 N
mu_nu = 1.0000000000000000e+00
Tmax_nu = 1.5501291679459377e+00
c_nu = 1.8398794991416076e+00
x0_nu = [1.00863061 0.         0.         0.23815669 0.47631338 0.65493089]
xf_nu = [0.96051596 0.19474068 0.1987088  0.         0.         0.        ]
||rf|| / R_M = 1.0000000102770787e+00


In [4]:
import twobody_RV_hamilton_log_lunar as lunar_model
ta = lunar_model.get_integrator(compact=False)

In [5]:
integration_max_steps = 120
q_penalty_coefficient = 10.0
# Note: Excluding the state transition matrix.

# =========================
# Set parameters for heyoka integrator
# =========================
# [modify-lunar] The lunar notebook currently uses the strict logarithmic Cartesian model.


# =========================
# Strict reduced Hamiltonian
# =========================
def strict_reduced_hamiltonian(state, eps, lambda0):
    """Evaluate the strict logarithmic-reduced nondimensional Hamiltonian."""
    # [modify-lunar] This matches twobody_RV_hamilton_log_lunar.py exactly.
    state = np.asarray(state, dtype=float)
    r = state[0:3]
    v = state[3:6]
    mass = state[6]
    lam_r = state[7:10]
    lam_v = state[10:13]
    lam_m = state[13]

    r_norm = np.linalg.norm(r)
    lam_v_norm = np.linalg.norm(lam_v)
    gravity = -mu_nu * r / r_norm**3

    rho_switch = 1.0 - (c2 * lam_v_norm / mass + lam_m) / lambda0
    u_mid = 0.5 * (u_min + u_max)
    u_half_range = 0.5 * (u_max - u_min)

    H0 = np.dot(lam_r, v) + np.dot(lam_v, gravity)
    H_log = H0 + lambda0 * c1 / c2 * (
        u_mid * rho_switch
        - u_half_range * np.sqrt(rho_switch**2 + 4.0*eps**2)
        + (u_max - u_min) * eps * (1.0 + np.log(
            (np.sqrt(rho_switch**2 + 4.0*eps**2) + 2.0*eps) / eps))
    )
    return float(H_log)


# [modify-lunar] Keep Minpack from crashing when it proposes non-finite trial points.
BAD_RESIDUAL = 1.0e20
TF_MIN_ALLOWED = t0_nu + 1.0e-10
TF_MAX_ALLOWED = t0_nu + 4.0


def _bad_residual(n):
    # [modify-lunar] Return finite values so scipy.root can fail gracefully.
    return np.full(n, BAD_RESIDUAL, dtype=float)


def _is_valid_tf(tf_nu):
    # [modify-lunar] The free terminal time is still an unconstrained Minpack variable,
    # so guard the propagation call against NaN/Inf/negative guesses.
    return np.isfinite(tf_nu) and (tf_nu > TF_MIN_ALLOWED) and (tf_nu <= TF_MAX_ALLOWED)


def _set_and_propagate(initial_costates, tf_nu, eps, lambda0):
    """Set the lunar parameters/state and propagate to a free terminal time."""
    # [modify-lunar] Parameter vector now includes u_min and u_max.
    ta.pars[:] = [mu_nu, c1, c2, eps, lambda0, u_min, u_max]
    ta.time = t0_nu
    ta.state[:7] = x0_nu + [m0_nu]
    ta.state[7:14] = initial_costates
    # [modify-lunar] tf_nu is an optimization unknown, not a global fixed value.
    result = ta.propagate_until(tf_nu, max_steps=integration_max_steps)
    if result[0] != hy.taylor_outcome.time_limit:
        raise RuntimeError('The integration did not reach its terminal time.')


def _lunar_physical_residual(eps, lambda0):
    """Return the 8 free-final-time lunar shooting residuals."""
    # [modify-lunar] 6 fixed terminal RV conditions + lambda_m(tf)=0 + H_log(tf)=0.
    Hf = strict_reduced_hamiltonian(ta.state[:14], eps, lambda0)
    return np.array([
        ta.state[0] - xf_nu[0],
        ta.state[1] - xf_nu[1],
        ta.state[2] - xf_nu[2],
        ta.state[3] - xf_nu[3],
        ta.state[4] - xf_nu[4],
        ta.state[5] - xf_nu[5],
        ta.state[13],
        Hf,
    ], dtype=float)


# =========================
# Define the shooting functions
# =========================
def shooting(z, eps=1.0e-5, lambda0=1.0, out=False, Time_out=None):
    # [modify-lunar] z[0:7] = initial costates; z[7] = free terminal time.
    z = np.asarray(z, dtype=float)
    if z.size != 8 or not np.all(np.isfinite(z)):
        return _bad_residual(8)

    initial_costates = z[:7]
    tf_nu = float(z[7])
    if not _is_valid_tf(tf_nu):
        return _bad_residual(8)

    try:
        _set_and_propagate(initial_costates, tf_nu, eps, lambda0)
        retval = _lunar_physical_residual(eps, lambda0)
        if not np.all(np.isfinite(retval)):
            return _bad_residual(8)
    except (ValueError, RuntimeError, OverflowError, FloatingPointError):
        return _bad_residual(8)

    if out:
        print(f"  Final Mass: {ta.state[6] * MUnit:.12f} kg")
        print(f"  Final Time: {tf_nu * TUnit:.12f} s  ({tf_nu:.12f} TU)")
        print(f"  H_log(tf): {retval[-1]:.16e}")

    if Time_out is not None:
        test = Time_out

    return retval


def shooting_augmented(
    z_augmented,
    logeps0=-0.0,
    logepsf=-5.0,
    lambda0=1.0,
    out=False,
    Time_out=None,
):
    # [modify-lunar] z_augmented = [7 initial costates, tf, normalized homotopy q].
    z_augmented = np.asarray(z_augmented, dtype=float)
    if z_augmented.size != 9 or not np.all(np.isfinite(z_augmented)):
        return _bad_residual(9)

    initial_costates = z_augmented[:7]
    tf_nu = float(z_augmented[7])
    q = float(z_augmented[8])
    if not _is_valid_tf(tf_nu):
        return _bad_residual(9)

    logeps = logeps0 + q * (logepsf - logeps0)
    eps = 10.0**logeps

    try:
        _set_and_propagate(initial_costates, tf_nu, eps, lambda0)
        Phi = _lunar_physical_residual(eps, lambda0)
        if not np.all(np.isfinite(Phi)):
            return _bad_residual(9)
    except (ValueError, RuntimeError, OverflowError, FloatingPointError):
        return _bad_residual(9)

    # [modify-lunar] 8 physical residuals + target-epsilon residual = 9 equations.
    retval = np.concatenate((Phi, [logeps - logepsf]))

    if out:
        print(f"  Final Mass: {ta.state[6] * MUnit:.12f} kg")
        print(f"  Final Time: {tf_nu * TUnit:.12f} s  ({tf_nu:.12f} TU)")
        print(f"  Final eps : {eps:.16e}  (logeps = {logeps:.8f}, q = {q:.8f})")
        print(f"  H_log(tf): {Phi[-1]:.16e}")

    if Time_out is not None:
        test = Time_out

    return retval


def shooting_augmented_gate(
    z_augmented,
    logeps0=0.0,
    logepsf=-5.0,
    rho0=1.0e-2,
    delta_rho=1.0e-8,
    gate_p=2.0,
    gate_kind="gaussian",
    lambda0=1.0,
    out=False,
    Time_out=None,
):
    # [modify-lunar] z_augmented = [7 initial costates, tf, normalized homotopy q].
    z_augmented = np.asarray(z_augmented, dtype=float)
    if z_augmented.size != 9 or not np.all(np.isfinite(z_augmented)):
        return _bad_residual(9)

    initial_costates = z_augmented[:7]
    tf_nu = float(z_augmented[7])
    q = float(z_augmented[8])
    if not _is_valid_tf(tf_nu):
        return _bad_residual(9)

    logeps = logeps0 + q * (logepsf - logeps0)
    eps = 10.0**logeps

    try:
        _set_and_propagate(initial_costates, tf_nu, eps, lambda0)
        Phi = _lunar_physical_residual(eps, lambda0)
        if not np.all(np.isfinite(Phi)):
            return _bad_residual(9)
    except (ValueError, RuntimeError, OverflowError, FloatingPointError):
        return _bad_residual(9)

    # [modify-lunar] Gate uses the RMS of all 8 lunar physical residuals,
    # including lambda_m(tf) and the strict free-time H_log(tf) residual.
    rho_residual = np.sqrt(delta_rho**2 + np.mean(Phi * Phi))

    if gate_kind == "gaussian":
        w = np.exp(- (rho_residual / rho0) ** gate_p)
    elif gate_kind == "rational":
        w = 1.0 / (1.0 + (rho_residual / rho0) ** gate_p)
    else:
        raise ValueError(f"Unknown gate_kind: {gate_kind}")

    penalty_q = q_penalty_coefficient * (max(q - 1.0, 0.0)**2 - max(-q, 0.0)**2)
    retval = np.concatenate((Phi, [q - w + penalty_q]))

    if out:
        print(f"  Final Mass: {ta.state[6] * MUnit:.12f} kg")
        print(f"  Final Time: {tf_nu * TUnit:.12f} s  ({tf_nu:.12f} TU)")
        print(
            f"  Final eps : {eps:.16e}  "
            f"(logeps = {logeps:.8f}, q = {q:.8f}, rho = {rho_residual:.3e})"
        )
        print(f"  H_log(tf): {Phi[-1]:.16e}")

    if Time_out is not None:
        test = Time_out

    return retval


# =========================
# Set parameters for minpack
# =========================
import time
from scipy.optimize import root

repeat_num_minpack = 5000
# repeat_num_minpack = 1

minpack_tol = 1.0e-10
minpack_xtol = 1.0e-10
minpack_maxfev = 50000
minpack_factor = 0.1
# [modify-lunar] Base unknown vector: 7 initial costates + free terminal time.
minpack_diag = [1.0] * 7 + [1.0]
minpack_eps = 0.0
minpack_error = 1.0e-7

np.set_printoptions(precision=16, suppress=True)


In [6]:
# =========================
# Common random initial guess pool
# =========================
# [modify-lunar] Generate all random initial guesses once so all methods are compared
# [modify-lunar] with exactly the same base guesses.
rng = np.random.default_rng(322)  # [modify-lunar]

# [modify-lunar] Base unknown vector: [7 initial costates, free terminal time].
tf_nu_min = 0.5
tf_nu_max = TF_MAX_ALLOWED

initial_guess_pool = np.empty((repeat_num_minpack, 8), dtype=float)  # [modify-lunar]
initial_guess_pool[:, :7] = (rng.random((repeat_num_minpack, 7)) - 0.5) * 2.0  # [modify-lunar]
initial_guess_pool[:, 6] = rng.random(repeat_num_minpack)  # [modify-lunar] lambda_m(0) in (0, 1).
initial_guess_pool[:, 7] = rng.random(repeat_num_minpack) * (tf_nu_max-tf_nu_min) + tf_nu_min  # [modify-lunar]

# [modify-lunar] Augmented methods use the same base guesses plus q0 = 0.
augmented_initial_guess_pool = np.empty((repeat_num_minpack, 9), dtype=float)  # [modify-lunar]
augmented_initial_guess_pool[:, :8] = initial_guess_pool  # [modify-lunar]
augmented_initial_guess_pool[:, 8] = 0.0  # [modify-lunar]

print(f"Generated {repeat_num_minpack} shared lunar initial guesses.")
print(f"tf guess range: [{initial_guess_pool[:, 7].min():.6f}, {initial_guess_pool[:, 7].max():.6f}] TU")
print(f"lambda_m(0) guess range: [{initial_guess_pool[:, 6].min():.6f}, {initial_guess_pool[:, 6].max():.6f}]")


Generated 5000 shared lunar initial guesses.
tf guess range: [0.500127, 3.999493] TU
lambda_m(0) guess range: [0.000197, 0.999722]


In [7]:

def prepare_timing():
    set_timing_priority()
    if sys.platform != 'darwin':
        return
    started = time.perf_counter()
    deadline = started + 0.25
    cpu = ctypes.c_size_t()
    while True:
        lib.pthread_cpu_number_np(ctypes.byref(cpu))
        if cpu.value in fast_cpus and time.perf_counter()-started >= 0.05:
            return
        if time.perf_counter() > deadline:
            raise RuntimeError('No Super-core scheduling during timing preparation.')


## Single shooting, epsilon=1


In [9]:
# =========================
# Initialize solving time and global optimal solution.
# =========================
epsf = 1.0e0
# epsf = 1.0e-5
lambda0 = 1.0

count_solved = 0
total_time = 0.0
total_nfev = 0
converge_nfev = 0

global_lambda = None
global_mf = -1.0

# =========================
# Shared initial guess loop
# =========================


In [10]:
records = []
for i in range(repeat_num_minpack):
    accepted_before = count_solved
    nfev_before = total_nfev
    r = None
    # [modify-lunar] Use the shared base initial guess: [lambda(0), tf].
    x0 = initial_guess_pool[i].copy()  # [modify-lunar]

    time_start = time.perf_counter()

    r = root(
        shooting,
        x0,
        args=(epsf, lambda0, False, None),
        method='hybr',
        tol=minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            'diag': minpack_diag,
            'eps': minpack_eps,
        }
    )

    time_end = time.perf_counter()
    total_time += time_end - time_start
    total_nfev += r['nfev']

    res = np.linalg.norm(r['fun'])
    if r['success'] and res < minpack_error:
        count_solved += 1
        converge_nfev += r['nfev']

        print(f"counter: {count_solved}/{i} nfev: {r['nfev']}", end='', flush=True)
        shooting(r['x'], epsf, lambda0, out=True, Time_out=None)

        final_mass = ta.state[6] * MUnit
        if final_mass > global_mf:
            global_mf = final_mass
            global_lambda = r['x']
    save_trial(i, r, count_solved > accepted_before, total_nfev - nfev_before)


# =========================
# Output convergence rate and average convergence time
# =========================
if count_solved < 1:
    print("Fail to solve the problem!")
else:
    print("Succeed to solve the problem!")
    print(f"Solved: {count_solved} in {repeat_num_minpack} ({count_solved/repeat_num_minpack:.4f}), terminal mass: {global_mf:.4f}kg")
    print(f"Average number of function calls for Minpack: {total_nfev/repeat_num_minpack:.0f}")
    print(f"Average number of function calls for convergent solution: {converge_nfev/count_solved:.0f}")
    # [modify-lunar] global_lambda contains [lambda(0), tf].
    print("global_lambda_tf:", global_lambda)
    print(f"global tf: {global_lambda[7] * TUnit:.12f} s")
records_by_method['Single_eps1'] = records


Succeed to solve the problem!
Solved: 1170 in 5000 (0.2340), terminal mass: 495.0675kg
Average number of function calls for Minpack: 42
Average number of function calls for convergent solution: 75
global_lambda_tf: [2.2276119784966233, -0.5348315977568402, 0.3069821821709586, 0.5013074923723758, 0.03896135355216771, 0.42241488085078094, 0.8905143789477997, 0.6680264806656917]
global tf: 691.267664855346 s


In [11]:
# =========================
# Computational time: all / successful / failed attempts
# =========================
timing_ids = np.arange(500)
accepted = np.array([r['accepted'] for r in records_by_method['Single_eps1']], dtype=bool)
successful_ids = timing_ids[accepted[timing_ids]]
failed_ids = timing_ids[~accepted[timing_ids]]
cpu_samples.clear()


In [12]:
%%timeit -n 1 -r 7 -v timing_all prepare_timing()
for i in timing_ids:
    sample_cpu()
    # [modify-lunar] Use the shared base initial guess: [lambda(0), tf].
    x0 = initial_guess_pool[i].copy()  # [modify-lunar]


    r = root(
        shooting,
        x0,
        args=(epsf, lambda0, False, None),
        method='hybr',
        tol=minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            'diag': minpack_diag,
            'eps': minpack_eps,
        }
    )


1.04 s ± 2.61 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [13]:
%%timeit -n 1 -r 7 -v timing_successful prepare_timing()
for i in successful_ids:
    sample_cpu()
    # [modify-lunar] Use the shared base initial guess: [lambda(0), tf].
    x0 = initial_guess_pool[i].copy()  # [modify-lunar]


    r = root(
        shooting,
        x0,
        args=(epsf, lambda0, False, None),
        method='hybr',
        tol=minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            'diag': minpack_diag,
            'eps': minpack_eps,
        }
    )


143 ms ± 2.79 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [14]:
%%timeit -n 1 -r 7 -v timing_failed prepare_timing()
for i in failed_ids:
    sample_cpu()
    # [modify-lunar] Use the shared base initial guess: [lambda(0), tf].
    x0 = initial_guess_pool[i].copy()  # [modify-lunar]


    r = root(
        shooting,
        x0,
        args=(epsf, lambda0, False, None),
        method='hybr',
        tol=minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            'diag': minpack_diag,
            'eps': minpack_eps,
        }
    )


904 ms ± 3.92 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [15]:
all_ms = timing_all.average * 1000.0 / len(timing_ids)
p_success = accepted.mean()
results_by_method['Single_eps1'] = dict(
    method='Single shooting, epsilon=1', trials=len(accepted), accepted=int(accepted.sum()),
    first_ms=all_ms / p_success if p_success else float('inf'),
    successful_ms=timing_successful.average * 1000.0 / len(successful_ids) if len(successful_ids) else float('nan'),
    failed_ms=timing_failed.average * 1000.0 / len(failed_ids) if len(failed_ids) else float('nan'),
    all_ms=all_ms,
    nfev_successful=float(np.mean([r['nfev'] for r in records_by_method['Single_eps1'] if r['accepted']])) if accepted.any() else float('nan'),
    nfev_all=float(np.mean([r['nfev'] for r in records_by_method['Single_eps1']])),
    timing_trials=len(timing_ids), timing_successful_trials=len(successful_ids), timing_failed_trials=len(failed_ids),
    timeit_runs_seconds=timing_all.all_runs,
    timeit_successful_seconds=timing_successful.all_runs,
    timeit_failed_seconds=timing_failed.all_runs,
    cpu_samples=dict(Counter(cpu_samples)),
    fast_core_sample_pct=100.0 * sum(cpu in fast_cpus for cpu in cpu_samples) / len(cpu_samples) if cpu_samples else None)


## Single shooting, epsilon=1e-5


In [16]:
# =========================
# Initialize solving time and global optimal solution.
# =========================
epsf = 1.0e-5
# epsf = 1.0e-5
lambda0 = 1.0

count_solved = 0
total_time = 0.0
total_nfev = 0
converge_nfev = 0

global_lambda = None
global_mf = -1.0

# =========================
# Shared initial guess loop
# =========================


In [17]:
records = []
for i in range(repeat_num_minpack):
    accepted_before = count_solved
    nfev_before = total_nfev
    r = None
    # [modify-lunar] Use the shared base initial guess: [lambda(0), tf].
    x0 = initial_guess_pool[i].copy()  # [modify-lunar]

    time_start = time.perf_counter()

    r = root(
        shooting,
        x0,
        args=(epsf, lambda0, False, None),
        method='hybr',
        tol=minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            'diag': minpack_diag,
            'eps': minpack_eps,
        }
    )

    time_end = time.perf_counter()
    total_time += time_end - time_start
    total_nfev += r['nfev']

    res = np.linalg.norm(r['fun'])
    if r['success'] and res < minpack_error:
        count_solved += 1
        converge_nfev += r['nfev']

        print(f"counter: {count_solved}/{i} nfev: {r['nfev']}", end='', flush=True)
        shooting(r['x'], epsf, lambda0, out=True, Time_out=None)

        final_mass = ta.state[6] * MUnit
        if final_mass > global_mf:
            global_mf = final_mass
            global_lambda = r['x']
    save_trial(i, r, count_solved > accepted_before, total_nfev - nfev_before)


# =========================
# Output convergence rate and average convergence time
# =========================
if count_solved < 1:
    print("Fail to solve the problem!")
else:
    print("Succeed to solve the problem!")
    print(f"Solved: {count_solved} in {repeat_num_minpack} ({count_solved/repeat_num_minpack:.4f}), terminal mass: {global_mf:.4f}kg")
    print(f"Average number of function calls for Minpack: {total_nfev/repeat_num_minpack:.0f}")
    print(f"Average number of function calls for convergent solution: {converge_nfev/count_solved:.0f}")
    # [modify-lunar] global_lambda contains [lambda(0), tf].
    print("global_lambda_tf:", global_lambda)
    print(f"global tf: {global_lambda[7] * TUnit:.12f} s")
records_by_method['Single_eps1e5'] = records


Succeed to solve the problem!
Solved: 788 in 5000 (0.1576), terminal mass: 510.4080kg
Average number of function calls for Minpack: 44
Average number of function calls for convergent solution: 106
global_lambda_tf: [0.9470997361403705, -0.1522939657616898, 0.2875555671178039, 0.20158655373246157, 0.08809919783960107, 0.3201581538437561, 0.45531025835644007, 0.6695497279500351]
global tf: 692.843907150705 s


In [18]:
# =========================
# Computational time: all / successful / failed attempts
# =========================
timing_ids = np.arange(500)
accepted = np.array([r['accepted'] for r in records_by_method['Single_eps1e5']], dtype=bool)
successful_ids = timing_ids[accepted[timing_ids]]
failed_ids = timing_ids[~accepted[timing_ids]]
cpu_samples.clear()


In [19]:
%%timeit -n 1 -r 7 -v timing_all prepare_timing()
for i in timing_ids:
    sample_cpu()
    # [modify-lunar] Use the shared base initial guess: [lambda(0), tf].
    x0 = initial_guess_pool[i].copy()  # [modify-lunar]


    r = root(
        shooting,
        x0,
        args=(epsf, lambda0, False, None),
        method='hybr',
        tol=minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            'diag': minpack_diag,
            'eps': minpack_eps,
        }
    )


1.34 s ± 4.74 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [20]:
%%timeit -n 1 -r 7 -v timing_successful prepare_timing()
for i in successful_ids:
    sample_cpu()
    # [modify-lunar] Use the shared base initial guess: [lambda(0), tf].
    x0 = initial_guess_pool[i].copy()  # [modify-lunar]


    r = root(
        shooting,
        x0,
        args=(epsf, lambda0, False, None),
        method='hybr',
        tol=minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            'diag': minpack_diag,
            'eps': minpack_eps,
        }
    )


333 ms ± 3.92 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [21]:
%%timeit -n 1 -r 7 -v timing_failed prepare_timing()
for i in failed_ids:
    sample_cpu()
    # [modify-lunar] Use the shared base initial guess: [lambda(0), tf].
    x0 = initial_guess_pool[i].copy()  # [modify-lunar]


    r = root(
        shooting,
        x0,
        args=(epsf, lambda0, False, None),
        method='hybr',
        tol=minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            'diag': minpack_diag,
            'eps': minpack_eps,
        }
    )


1.05 s ± 52.8 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [22]:
all_ms = timing_all.average * 1000.0 / len(timing_ids)
p_success = accepted.mean()
results_by_method['Single_eps1e5'] = dict(
    method='Single shooting, epsilon=1e-5', trials=len(accepted), accepted=int(accepted.sum()),
    first_ms=all_ms / p_success if p_success else float('inf'),
    successful_ms=timing_successful.average * 1000.0 / len(successful_ids) if len(successful_ids) else float('nan'),
    failed_ms=timing_failed.average * 1000.0 / len(failed_ids) if len(failed_ids) else float('nan'),
    all_ms=all_ms,
    nfev_successful=float(np.mean([r['nfev'] for r in records_by_method['Single_eps1e5'] if r['accepted']])) if accepted.any() else float('nan'),
    nfev_all=float(np.mean([r['nfev'] for r in records_by_method['Single_eps1e5']])),
    timing_trials=len(timing_ids), timing_successful_trials=len(successful_ids), timing_failed_trials=len(failed_ids),
    timeit_runs_seconds=timing_all.all_runs,
    timeit_successful_seconds=timing_successful.all_runs,
    timeit_failed_seconds=timing_failed.all_runs,
    cpu_samples=dict(Counter(cpu_samples)),
    fast_core_sample_pct=100.0 * sum(cpu in fast_cpus for cpu in cpu_samples) / len(cpu_samples) if cpu_samples else None)


## Stepwise


In [23]:
# =========================
# Initialize solving time and global optimal solution.
# =========================
eps0 = 1.0e0
epsf = 1.0e-5

lambda0 = 1.0

count_solved = 0
total_time = 0.0
total_nfev = 0
converge_nfev = 0

global_lambda = None
global_mf = -1.0

global_eps_seq = None
global_mf_seq = None
global_lambda_seq = None

d0 = 0.1
d_shrink = 0.8
d_recover = 0.8
d_max = 0.999
max_backtrack = 20
max_total_steps = 300


def solve_for_eps(x_init, eps_val):
    time_start = time.perf_counter()

    r = root(
        shooting,
        x_init,
        args=(eps_val, lambda0, False, None),
        method='hybr',
        tol=minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            'diag': minpack_diag,
            'eps': minpack_eps,
        }
    )

    elapsed = time.perf_counter() - time_start
    res = np.linalg.norm(r['fun'])
    ok = r['success'] and res < minpack_error
    return r, ok, res, elapsed


# =========================
# Shared initial guess loop
# =========================


In [24]:
records = []
for i in range(repeat_num_minpack):
    accepted_before = count_solved
    nfev_before = total_nfev
    r = None
    r_next = None
    # [modify-lunar] Use the shared base initial guess: [lambda(0), tf].
    x0 = initial_guess_pool[i].copy()  # [modify-lunar]

    this_time = 0.0
    this_nfev = 0

    eps_curr = eps0
    d = d0
    x_guess = x0.copy()

    eps_seq = []
    mf_seq = []
    lambda_seq = []

    solved_this_outer = False

    # ============================================================
    # 0) Solve the easiest problem at eps0 once
    # ============================================================
    r0, ok0, res0, elapsed0 = solve_for_eps(x_guess, eps_curr)
    this_time += elapsed0
    this_nfev += r0['nfev']

    if ok0:
        x_guess = r0['x']

        shooting(x_guess, eps_curr, lambda0, out=False, Time_out=None)
        mf_curr = ta.state[6] * MUnit

        eps_seq.append(eps_curr)
        mf_seq.append(mf_curr)
        lambda_seq.append(x_guess.copy())

    # ============================================================
    # 1) Continuation: advance eps until epsf
    # ============================================================
    if ok0:
        for step in range(max_total_steps):
            if eps_curr <= epsf:
                solved_this_outer = True

                final_mass = mf_seq[-1]
                if final_mass > global_mf:
                    global_mf = final_mass
                    global_lambda = lambda_seq[-1]
                    global_eps_seq = eps_seq.copy()
                    global_mf_seq = mf_seq.copy()
                    global_lambda_seq = [v.copy() for v in lambda_seq]

                break

            backtrack = 0
            advanced = False

            while backtrack <= max_backtrack:
                eps_next = max(eps_curr * d, epsf)

                if eps_next == eps_curr:
                    eps_next = epsf

                r_next, ok_next, res_next, elapsed_next = solve_for_eps(x_guess, eps_next)
                this_time += elapsed_next
                this_nfev += r_next['nfev']

                if ok_next:
                    eps_curr = eps_next
                    x_guess = r_next['x']
                    advanced = True

                    shooting(x_guess, eps_curr, lambda0, out=False, Time_out=None)
                    mf_curr = ta.state[6] * MUnit

                    eps_seq.append(eps_curr)
                    mf_seq.append(mf_curr)
                    lambda_seq.append(x_guess.copy())

                    d = max(d0, d * d_recover)
                    break
                else:
                    d = min(d_max, d / d_shrink)
                    backtrack += 1

            if not advanced:
                break

    total_time += this_time
    total_nfev += this_nfev

    if solved_this_outer:
        count_solved += 1
        converge_nfev += this_nfev

        print(
            f"counter: {count_solved}/{i+1} "
            f"steps(accepted): {len(eps_seq)} "
            f"nfev(sum): {this_nfev} "
            f"eps_end: {eps_seq[-1]:.1e} "
            f"mf_end: {mf_seq[-1]:.4f}kg",
            end='\n',
            flush=True
                )
    save_trial(i, r_next if ok0 and r_next is not None else r0, count_solved > accepted_before, total_nfev - nfev_before)


# =========================
# Output convergence rate and average convergence time
# =========================
if count_solved < 1:
    print("Fail to solve the problem!")
else:
    print("Succeed to solve the problem!")
    print(f"Solved: {count_solved} in {repeat_num_minpack} ({count_solved/repeat_num_minpack:.4f}), terminal mass: {global_mf:.4f}kg")
    print(f"Average number of function calls for Minpack: {total_nfev/repeat_num_minpack:.0f}")
    print(f"Average number of function calls for convergent solution: {converge_nfev/count_solved:.0f}")

    # [modify-lunar] Each stored vector contains [lambda(0), tf].
    print("global_lambda_tf (at epsf):", global_lambda)
    print(f"global tf: {global_lambda[7] * TUnit:.12f} s")
    print("global_eps_seq:", global_eps_seq)
    print("global_mf_seq:", global_mf_seq)
    print("global_lambda_seq length:", None if global_lambda_seq is None else len(global_lambda_seq))
records_by_method['Homotopy'] = records


Succeed to solve the problem!
Solved: 1170 in 5000 (0.2340), terminal mass: 510.4080kg
Average number of function calls for Minpack: 70
Average number of function calls for convergent solution: 197
global_lambda_tf (at epsf): [0.9470997361374511, -0.15229396575593943, 0.28755556711684044, 0.20158655373131748, 0.08809919784234127, 0.3201581538444129, 0.45531025835986494, 0.6695497279508392]
global tf: 692.843907151537 s
global_eps_seq: [1.0, 0.1, 0.010000000000000002, 0.0010000000000000002, 0.00010000000000000003, 1.0000000000000004e-05, 1e-05]
global_mf_seq: [495.06749411264536, 501.01173352119446, 508.1394688649407, 510.1270479691772, 510.3815197847637, 510.4079983281988, 510.4079983282313]
global_lambda_seq length: 7


In [25]:
# =========================
# Computational time: all / successful / failed attempts
# =========================
timing_ids = np.arange(500)
accepted = np.array([r['accepted'] for r in records_by_method['Homotopy']], dtype=bool)
successful_ids = timing_ids[accepted[timing_ids]]
failed_ids = timing_ids[~accepted[timing_ids]]
cpu_samples.clear()


In [26]:
%%timeit -n 1 -r 7 -v timing_all prepare_timing()
# =========================
# Initialize solving time and global optimal solution.
# =========================
eps0 = 1.0e0
epsf = 1.0e-5

lambda0 = 1.0

count_solved = 0
total_time = 0.0
total_nfev = 0
converge_nfev = 0

global_lambda = None
global_mf = -1.0

global_eps_seq = None
global_mf_seq = None
global_lambda_seq = None

d0 = 0.1
d_shrink = 0.8
d_recover = 0.8
d_max = 0.999
max_backtrack = 20
max_total_steps = 300


def solve_for_eps(x_init, eps_val):
    time_start = time.perf_counter()

    r = root(
        shooting,
        x_init,
        args=(eps_val, lambda0, False, None),
        method='hybr',
        tol=minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            'diag': minpack_diag,
            'eps': minpack_eps,
        }
    )

    elapsed = time.perf_counter() - time_start
    res = np.linalg.norm(r['fun'])
    ok = r['success'] and res < minpack_error
    return r, ok, res, elapsed


# =========================
# Shared initial guess loop
# =========================

for i in timing_ids:
    sample_cpu()
    # [modify-lunar] Use the shared base initial guess: [lambda(0), tf].
    x0 = initial_guess_pool[i].copy()  # [modify-lunar]

    this_time = 0.0
    this_nfev = 0

    eps_curr = eps0
    d = d0
    x_guess = x0.copy()

    eps_seq = []
    mf_seq = []
    lambda_seq = []

    solved_this_outer = False

    # ============================================================
    # 0) Solve the easiest problem at eps0 once
    # ============================================================
    r0, ok0, res0, elapsed0 = solve_for_eps(x_guess, eps_curr)
    this_time += elapsed0
    this_nfev += r0['nfev']

    if ok0:
        x_guess = r0['x']

        shooting(x_guess, eps_curr, lambda0, out=False, Time_out=None)
        mf_curr = ta.state[6] * MUnit

        eps_seq.append(eps_curr)
        mf_seq.append(mf_curr)
        lambda_seq.append(x_guess.copy())

    # ============================================================
    # 1) Continuation: advance eps until epsf
    # ============================================================
    if ok0:
        for step in range(max_total_steps):
            if eps_curr <= epsf:
                solved_this_outer = True

                final_mass = mf_seq[-1]
                if final_mass > global_mf:
                    global_mf = final_mass
                    global_lambda = lambda_seq[-1]
                    global_eps_seq = eps_seq.copy()
                    global_mf_seq = mf_seq.copy()
                    global_lambda_seq = [v.copy() for v in lambda_seq]

                break

            backtrack = 0
            advanced = False

            while backtrack <= max_backtrack:
                eps_next = max(eps_curr * d, epsf)

                if eps_next == eps_curr:
                    eps_next = epsf

                r_next, ok_next, res_next, elapsed_next = solve_for_eps(x_guess, eps_next)
                this_time += elapsed_next
                this_nfev += r_next['nfev']

                if ok_next:
                    eps_curr = eps_next
                    x_guess = r_next['x']
                    advanced = True

                    shooting(x_guess, eps_curr, lambda0, out=False, Time_out=None)
                    mf_curr = ta.state[6] * MUnit

                    eps_seq.append(eps_curr)
                    mf_seq.append(mf_curr)
                    lambda_seq.append(x_guess.copy())

                    d = max(d0, d * d_recover)
                    break
                else:
                    d = min(d_max, d / d_shrink)
                    backtrack += 1

            if not advanced:
                break

    total_time += this_time
    total_nfev += this_nfev

    if solved_this_outer:
        count_solved += 1
        converge_nfev += this_nfev


1.76 s ± 58.7 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [27]:
%%timeit -n 1 -r 7 -v timing_successful prepare_timing()
# =========================
# Initialize solving time and global optimal solution.
# =========================
eps0 = 1.0e0
epsf = 1.0e-5

lambda0 = 1.0

count_solved = 0
total_time = 0.0
total_nfev = 0
converge_nfev = 0

global_lambda = None
global_mf = -1.0

global_eps_seq = None
global_mf_seq = None
global_lambda_seq = None

d0 = 0.1
d_shrink = 0.8
d_recover = 0.8
d_max = 0.999
max_backtrack = 20
max_total_steps = 300


def solve_for_eps(x_init, eps_val):
    time_start = time.perf_counter()

    r = root(
        shooting,
        x_init,
        args=(eps_val, lambda0, False, None),
        method='hybr',
        tol=minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            'diag': minpack_diag,
            'eps': minpack_eps,
        }
    )

    elapsed = time.perf_counter() - time_start
    res = np.linalg.norm(r['fun'])
    ok = r['success'] and res < minpack_error
    return r, ok, res, elapsed


# =========================
# Shared initial guess loop
# =========================

for i in successful_ids:
    sample_cpu()
    # [modify-lunar] Use the shared base initial guess: [lambda(0), tf].
    x0 = initial_guess_pool[i].copy()  # [modify-lunar]

    this_time = 0.0
    this_nfev = 0

    eps_curr = eps0
    d = d0
    x_guess = x0.copy()

    eps_seq = []
    mf_seq = []
    lambda_seq = []

    solved_this_outer = False

    # ============================================================
    # 0) Solve the easiest problem at eps0 once
    # ============================================================
    r0, ok0, res0, elapsed0 = solve_for_eps(x_guess, eps_curr)
    this_time += elapsed0
    this_nfev += r0['nfev']

    if ok0:
        x_guess = r0['x']

        shooting(x_guess, eps_curr, lambda0, out=False, Time_out=None)
        mf_curr = ta.state[6] * MUnit

        eps_seq.append(eps_curr)
        mf_seq.append(mf_curr)
        lambda_seq.append(x_guess.copy())

    # ============================================================
    # 1) Continuation: advance eps until epsf
    # ============================================================
    if ok0:
        for step in range(max_total_steps):
            if eps_curr <= epsf:
                solved_this_outer = True

                final_mass = mf_seq[-1]
                if final_mass > global_mf:
                    global_mf = final_mass
                    global_lambda = lambda_seq[-1]
                    global_eps_seq = eps_seq.copy()
                    global_mf_seq = mf_seq.copy()
                    global_lambda_seq = [v.copy() for v in lambda_seq]

                break

            backtrack = 0
            advanced = False

            while backtrack <= max_backtrack:
                eps_next = max(eps_curr * d, epsf)

                if eps_next == eps_curr:
                    eps_next = epsf

                r_next, ok_next, res_next, elapsed_next = solve_for_eps(x_guess, eps_next)
                this_time += elapsed_next
                this_nfev += r_next['nfev']

                if ok_next:
                    eps_curr = eps_next
                    x_guess = r_next['x']
                    advanced = True

                    shooting(x_guess, eps_curr, lambda0, out=False, Time_out=None)
                    mf_curr = ta.state[6] * MUnit

                    eps_seq.append(eps_curr)
                    mf_seq.append(mf_curr)
                    lambda_seq.append(x_guess.copy())

                    d = max(d0, d * d_recover)
                    break
                else:
                    d = min(d_max, d / d_shrink)
                    backtrack += 1

            if not advanced:
                break

    total_time += this_time
    total_nfev += this_nfev

    if solved_this_outer:
        count_solved += 1
        converge_nfev += this_nfev


815 ms ± 9.54 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [28]:
%%timeit -n 1 -r 7 -v timing_failed prepare_timing()
# =========================
# Initialize solving time and global optimal solution.
# =========================
eps0 = 1.0e0
epsf = 1.0e-5

lambda0 = 1.0

count_solved = 0
total_time = 0.0
total_nfev = 0
converge_nfev = 0

global_lambda = None
global_mf = -1.0

global_eps_seq = None
global_mf_seq = None
global_lambda_seq = None

d0 = 0.1
d_shrink = 0.8
d_recover = 0.8
d_max = 0.999
max_backtrack = 20
max_total_steps = 300


def solve_for_eps(x_init, eps_val):
    time_start = time.perf_counter()

    r = root(
        shooting,
        x_init,
        args=(eps_val, lambda0, False, None),
        method='hybr',
        tol=minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            'diag': minpack_diag,
            'eps': minpack_eps,
        }
    )

    elapsed = time.perf_counter() - time_start
    res = np.linalg.norm(r['fun'])
    ok = r['success'] and res < minpack_error
    return r, ok, res, elapsed


# =========================
# Shared initial guess loop
# =========================

for i in failed_ids:
    sample_cpu()
    # [modify-lunar] Use the shared base initial guess: [lambda(0), tf].
    x0 = initial_guess_pool[i].copy()  # [modify-lunar]

    this_time = 0.0
    this_nfev = 0

    eps_curr = eps0
    d = d0
    x_guess = x0.copy()

    eps_seq = []
    mf_seq = []
    lambda_seq = []

    solved_this_outer = False

    # ============================================================
    # 0) Solve the easiest problem at eps0 once
    # ============================================================
    r0, ok0, res0, elapsed0 = solve_for_eps(x_guess, eps_curr)
    this_time += elapsed0
    this_nfev += r0['nfev']

    if ok0:
        x_guess = r0['x']

        shooting(x_guess, eps_curr, lambda0, out=False, Time_out=None)
        mf_curr = ta.state[6] * MUnit

        eps_seq.append(eps_curr)
        mf_seq.append(mf_curr)
        lambda_seq.append(x_guess.copy())

    # ============================================================
    # 1) Continuation: advance eps until epsf
    # ============================================================
    if ok0:
        for step in range(max_total_steps):
            if eps_curr <= epsf:
                solved_this_outer = True

                final_mass = mf_seq[-1]
                if final_mass > global_mf:
                    global_mf = final_mass
                    global_lambda = lambda_seq[-1]
                    global_eps_seq = eps_seq.copy()
                    global_mf_seq = mf_seq.copy()
                    global_lambda_seq = [v.copy() for v in lambda_seq]

                break

            backtrack = 0
            advanced = False

            while backtrack <= max_backtrack:
                eps_next = max(eps_curr * d, epsf)

                if eps_next == eps_curr:
                    eps_next = epsf

                r_next, ok_next, res_next, elapsed_next = solve_for_eps(x_guess, eps_next)
                this_time += elapsed_next
                this_nfev += r_next['nfev']

                if ok_next:
                    eps_curr = eps_next
                    x_guess = r_next['x']
                    advanced = True

                    shooting(x_guess, eps_curr, lambda0, out=False, Time_out=None)
                    mf_curr = ta.state[6] * MUnit

                    eps_seq.append(eps_curr)
                    mf_seq.append(mf_curr)
                    lambda_seq.append(x_guess.copy())

                    d = max(d0, d * d_recover)
                    break
                else:
                    d = min(d_max, d / d_shrink)
                    backtrack += 1

            if not advanced:
                break

    total_time += this_time
    total_nfev += this_nfev

    if solved_this_outer:
        count_solved += 1
        converge_nfev += this_nfev


944 ms ± 15.5 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [29]:
all_ms = timing_all.average * 1000.0 / len(timing_ids)
p_success = accepted.mean()
results_by_method['Homotopy'] = dict(
    method='Stepwise', trials=len(accepted), accepted=int(accepted.sum()),
    first_ms=all_ms / p_success if p_success else float('inf'),
    successful_ms=timing_successful.average * 1000.0 / len(successful_ids) if len(successful_ids) else float('nan'),
    failed_ms=timing_failed.average * 1000.0 / len(failed_ids) if len(failed_ids) else float('nan'),
    all_ms=all_ms,
    nfev_successful=float(np.mean([r['nfev'] for r in records_by_method['Homotopy'] if r['accepted']])) if accepted.any() else float('nan'),
    nfev_all=float(np.mean([r['nfev'] for r in records_by_method['Homotopy']])),
    timing_trials=len(timing_ids), timing_successful_trials=len(successful_ids), timing_failed_trials=len(failed_ids),
    timeit_runs_seconds=timing_all.all_runs,
    timeit_successful_seconds=timing_successful.all_runs,
    timeit_failed_seconds=timing_failed.all_runs,
    cpu_samples=dict(Counter(cpu_samples)),
    fast_core_sample_pct=100.0 * sum(cpu in fast_cpus for cpu in cpu_samples) / len(cpu_samples) if cpu_samples else None)


## Ungated


In [30]:
# =========================
# Initialize solving time and global optimal solution.
# =========================
eps0 = 1.0e-0
epsf = 1.0e-5
logeps0 = np.log10(eps0)
logepsf = np.log10(epsf)
lambda0 = 1.0

count_solved = 0
total_time = 0.0
total_nfev = 0
converge_nfev = 0

global_lambda = None
global_mf = -1.0

# =========================
# Shared initial guess loop
# =========================


In [31]:
records = []
for i in range(repeat_num_minpack):
    accepted_before = count_solved
    nfev_before = total_nfev
    r = None
    # [modify-lunar] Use the shared augmented initial guess: [lambda(0), tf, q0].
    x0 = augmented_initial_guess_pool[i].copy()  # [modify-lunar]

    time_start = time.perf_counter()

    r = root(
        shooting_augmented,
        x0,
        args=(logeps0, logepsf, lambda0, False, None),
        method='hybr',
        tol=minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            # [modify-lunar] 8 base unknowns plus q.
            'diag': minpack_diag + [1.0],
            'eps': minpack_eps,
        }
    )

    time_end = time.perf_counter()
    total_time += time_end - time_start
    total_nfev += r['nfev']

    res = np.linalg.norm(r['fun'])
    if r['success'] and res < minpack_error:
        count_solved += 1
        converge_nfev += r['nfev']

        print(f"counter: {count_solved}/{i} nfev: {r['nfev']}", end='', flush=True)
        shooting_augmented(r['x'], logeps0, logepsf, lambda0, out=True, Time_out=None)

        final_mass = ta.state[6] * MUnit
        if final_mass > global_mf:
            global_mf = final_mass
            global_lambda = r['x']
    save_trial(i, r, count_solved > accepted_before, total_nfev - nfev_before)


# =========================
# Output convergence rate and average convergence time
# =========================
if count_solved < 1:
    print("Fail to solve the problem!")
else:
    print("Succeed to solve the problem!")
    print(f"Solved: {count_solved} in {repeat_num_minpack} ({count_solved/repeat_num_minpack:.4f}), terminal mass: {global_mf:.4f}kg")
    print(f"Average number of function calls for Minpack: {total_nfev/repeat_num_minpack:.0f}")
    print(f"Average number of function calls for convergent solution: {converge_nfev/count_solved:.0f}")
    # [modify-lunar] global_lambda contains [lambda(0), tf, q].
    print("global_lambda_tf_q:", global_lambda)
    print(f"global tf: {global_lambda[7] * TUnit:.12f} s")
records_by_method['Ungated'] = records


Succeed to solve the problem!
Solved: 948 in 5000 (0.1896), terminal mass: 510.4080kg
Average number of function calls for Minpack: 54
Average number of function calls for convergent solution: 115
global_lambda_tf_q: [0.9470997361066289, -0.15229396575871565, 0.2875555671238803, 0.20158655372729248, 0.08809919783457427, 0.3201581538498681, 0.45531025834836725, 0.6695497279543094, 1.0]
global tf: 692.843907155128 s


In [32]:
# =========================
# Computational time: all / successful / failed attempts
# =========================
timing_ids = np.arange(500)
accepted = np.array([r['accepted'] for r in records_by_method['Ungated']], dtype=bool)
successful_ids = timing_ids[accepted[timing_ids]]
failed_ids = timing_ids[~accepted[timing_ids]]
cpu_samples.clear()


In [33]:
%%timeit -n 1 -r 7 -v timing_all prepare_timing()
for i in timing_ids:
    sample_cpu()
    # [modify-lunar] Use the shared augmented initial guess: [lambda(0), tf, q0].
    x0 = augmented_initial_guess_pool[i].copy()  # [modify-lunar]


    r = root(
        shooting_augmented,
        x0,
        args=(logeps0, logepsf, lambda0, False, None),
        method='hybr',
        tol=minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            # [modify-lunar] 8 base unknowns plus q.
            'diag': minpack_diag + [1.0],
            'eps': minpack_eps,
        }
    )


1.52 s ± 30.9 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [34]:
%%timeit -n 1 -r 7 -v timing_successful prepare_timing()
for i in successful_ids:
    sample_cpu()
    # [modify-lunar] Use the shared augmented initial guess: [lambda(0), tf, q0].
    x0 = augmented_initial_guess_pool[i].copy()  # [modify-lunar]


    r = root(
        shooting_augmented,
        x0,
        args=(logeps0, logepsf, lambda0, False, None),
        method='hybr',
        tol=minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            # [modify-lunar] 8 base unknowns plus q.
            'diag': minpack_diag + [1.0],
            'eps': minpack_eps,
        }
    )


467 ms ± 17.8 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [35]:
%%timeit -n 1 -r 7 -v timing_failed prepare_timing()
for i in failed_ids:
    sample_cpu()
    # [modify-lunar] Use the shared augmented initial guess: [lambda(0), tf, q0].
    x0 = augmented_initial_guess_pool[i].copy()  # [modify-lunar]


    r = root(
        shooting_augmented,
        x0,
        args=(logeps0, logepsf, lambda0, False, None),
        method='hybr',
        tol=minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            # [modify-lunar] 8 base unknowns plus q.
            'diag': minpack_diag + [1.0],
            'eps': minpack_eps,
        }
    )


1.04 s ± 10.5 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [36]:
all_ms = timing_all.average * 1000.0 / len(timing_ids)
p_success = accepted.mean()
results_by_method['Ungated'] = dict(
    method='Ungated', trials=len(accepted), accepted=int(accepted.sum()),
    first_ms=all_ms / p_success if p_success else float('inf'),
    successful_ms=timing_successful.average * 1000.0 / len(successful_ids) if len(successful_ids) else float('nan'),
    failed_ms=timing_failed.average * 1000.0 / len(failed_ids) if len(failed_ids) else float('nan'),
    all_ms=all_ms,
    nfev_successful=float(np.mean([r['nfev'] for r in records_by_method['Ungated'] if r['accepted']])) if accepted.any() else float('nan'),
    nfev_all=float(np.mean([r['nfev'] for r in records_by_method['Ungated']])),
    timing_trials=len(timing_ids), timing_successful_trials=len(successful_ids), timing_failed_trials=len(failed_ids),
    timeit_runs_seconds=timing_all.all_runs,
    timeit_successful_seconds=timing_successful.all_runs,
    timeit_failed_seconds=timing_failed.all_runs,
    cpu_samples=dict(Counter(cpu_samples)),
    fast_core_sample_pct=100.0 * sum(cpu in fast_cpus for cpu in cpu_samples) / len(cpu_samples) if cpu_samples else None)


## Option 1: Exp p=1, A=10


In [37]:
# =========================
# Initialize solving time and global optimal solution.
# =========================
eps0 = 1.0e-0
epsf = 1.0e-5
logeps0 = np.log10(eps0)
logepsf = np.log10(epsf)

# [MODIFIED] parameters for the augmented residual
gate_option = 'gaussian'
gate_p = 1.0
rho0 = 0.2330265204971175

# [other parameters]:
fd_eps = 0.
delta_rho = 1.0e-8


# [general settings]:
lambda0 = 1.0

count_solved = 0
total_time = 0.0
total_nfev = 0
converge_nfev = 0

global_lambda = None
global_mf = -1.0

# =========================
# Shared initial guess loop
# =========================


In [38]:
records = []
for i in range(repeat_num_minpack):
    accepted_before = count_solved
    nfev_before = total_nfev
    r = None
    # [modify-lunar] Use the shared augmented initial guess: [lambda(0), tf, q0].
    x0 = augmented_initial_guess_pool[i].copy()  # [modify-lunar]

    time_start = time.perf_counter()

    r = root(
        shooting_augmented_gate,
        x0,
        args=(logeps0, logepsf, rho0, delta_rho, gate_p, gate_option, lambda0, False, None),
        method='hybr',
        tol=minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            # [modify-lunar] 8 base unknowns plus q.
            'diag': minpack_diag + [1.0],
            'eps': fd_eps,
        }
    )

    time_end = time.perf_counter()
    total_time += time_end - time_start
    total_nfev += r['nfev']

    res = np.linalg.norm(r['fun'])
    if r['success'] and res < minpack_error:
        count_solved += 1
        converge_nfev += r['nfev']

        print(f"counter: {count_solved}/{i} nfev: {r['nfev']}", end='', flush=True)

        shooting_augmented_gate(
            r['x'],
            logeps0, logepsf, rho0, delta_rho, gate_p, gate_option, lambda0,
            out=True,
            Time_out=None,
        )

        final_mass = ta.state[6] * MUnit
        if final_mass > global_mf:
            global_mf = final_mass
            global_lambda = r['x']
    save_trial(i, r, count_solved > accepted_before, total_nfev - nfev_before)


# =========================
# Output convergence rate and average convergence time
# =========================
if count_solved < 1:
    print("Fail to solve the problem!")
else:
    print("Succeed to solve the problem!")
    print(f"Solved: {count_solved} in {repeat_num_minpack} ({count_solved/repeat_num_minpack:.4f}), terminal mass: {global_mf:.4f}kg")
    print(f"Average number of function calls for Minpack: {total_nfev/repeat_num_minpack:.0f}")
    print(f"Average number of function calls for convergent solution: {converge_nfev/count_solved:.0f}")
    # [modify-lunar] global_lambda contains [lambda(0), tf, q].
    print("global_lambda_tf_q:", global_lambda)
    print(f"global tf: {global_lambda[7] * TUnit:.12f} s")
records_by_method['Gated_Exp_p1'] = records


Succeed to solve the problem!
Solved: 1245 in 5000 (0.2490), terminal mass: 510.4080kg
Average number of function calls for Minpack: 54
Average number of function calls for convergent solution: 112
global_lambda_tf_q: [0.947099736192364, -0.152293965778252, 0.2875555671142557, 0.20158655374992665, 0.08809919783134722, 0.3201581538367128, 0.45531025836254313, 0.6695497279442808, 0.9999999570630352]
global tf: 692.843907144751 s


In [39]:
# =========================
# Computational time: all / successful / failed attempts
# =========================
timing_ids = np.arange(500)
accepted = np.array([r['accepted'] for r in records_by_method['Gated_Exp_p1']], dtype=bool)
successful_ids = timing_ids[accepted[timing_ids]]
failed_ids = timing_ids[~accepted[timing_ids]]
cpu_samples.clear()


In [40]:
%%timeit -n 1 -r 7 -v timing_all prepare_timing()
for i in timing_ids:
    sample_cpu()
    # [modify-lunar] Use the shared augmented initial guess: [lambda(0), tf, q0].
    x0 = augmented_initial_guess_pool[i].copy()  # [modify-lunar]


    r = root(
        shooting_augmented_gate,
        x0,
        args=(logeps0, logepsf, rho0, delta_rho, gate_p, gate_option, lambda0, False, None),
        method='hybr',
        tol=minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            # [modify-lunar] 8 base unknowns plus q.
            'diag': minpack_diag + [1.0],
            'eps': fd_eps,
        }
    )


1.58 s ± 20.6 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [41]:
%%timeit -n 1 -r 7 -v timing_successful prepare_timing()
for i in successful_ids:
    sample_cpu()
    # [modify-lunar] Use the shared augmented initial guess: [lambda(0), tf, q0].
    x0 = augmented_initial_guess_pool[i].copy()  # [modify-lunar]


    r = root(
        shooting_augmented_gate,
        x0,
        args=(logeps0, logepsf, rho0, delta_rho, gate_p, gate_option, lambda0, False, None),
        method='hybr',
        tol=minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            # [modify-lunar] 8 base unknowns plus q.
            'diag': minpack_diag + [1.0],
            'eps': fd_eps,
        }
    )


581 ms ± 1.31 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [42]:
%%timeit -n 1 -r 7 -v timing_failed prepare_timing()
for i in failed_ids:
    sample_cpu()
    # [modify-lunar] Use the shared augmented initial guess: [lambda(0), tf, q0].
    x0 = augmented_initial_guess_pool[i].copy()  # [modify-lunar]


    r = root(
        shooting_augmented_gate,
        x0,
        args=(logeps0, logepsf, rho0, delta_rho, gate_p, gate_option, lambda0, False, None),
        method='hybr',
        tol=minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            # [modify-lunar] 8 base unknowns plus q.
            'diag': minpack_diag + [1.0],
            'eps': fd_eps,
        }
    )


976 ms ± 19.4 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [43]:
all_ms = timing_all.average * 1000.0 / len(timing_ids)
p_success = accepted.mean()
results_by_method['Gated_Exp_p1'] = dict(
    method='Option 1: Exp p=1, A=10', trials=len(accepted), accepted=int(accepted.sum()),
    first_ms=all_ms / p_success if p_success else float('inf'),
    successful_ms=timing_successful.average * 1000.0 / len(successful_ids) if len(successful_ids) else float('nan'),
    failed_ms=timing_failed.average * 1000.0 / len(failed_ids) if len(failed_ids) else float('nan'),
    all_ms=all_ms,
    nfev_successful=float(np.mean([r['nfev'] for r in records_by_method['Gated_Exp_p1'] if r['accepted']])) if accepted.any() else float('nan'),
    nfev_all=float(np.mean([r['nfev'] for r in records_by_method['Gated_Exp_p1']])),
    timing_trials=len(timing_ids), timing_successful_trials=len(successful_ids), timing_failed_trials=len(failed_ids),
    timeit_runs_seconds=timing_all.all_runs,
    timeit_successful_seconds=timing_successful.all_runs,
    timeit_failed_seconds=timing_failed.all_runs,
    cpu_samples=dict(Counter(cpu_samples)),
    fast_core_sample_pct=100.0 * sum(cpu in fast_cpus for cpu in cpu_samples) / len(cpu_samples) if cpu_samples else None)


## Option 2: Rational p=1, A=10


In [44]:
# =========================
# Initialize solving time and global optimal solution.
# =========================
eps0 = 1.0e-0
epsf = 1.0e-5
logeps0 = np.log10(eps0)
logepsf = np.log10(epsf)

# [MODIFIED] parameters for the augmented residual
gate_option = 'rational'
gate_p = 1.0
rho0 = 0.018495327

# [other parameters]:
fd_eps = 0.
delta_rho = 1.0e-8


# [general settings]:
lambda0 = 1.0

count_solved = 0
total_time = 0.0
total_nfev = 0
converge_nfev = 0

global_lambda = None
global_mf = -1.0

# =========================
# Shared initial guess loop
# =========================


In [45]:
records = []
for i in range(repeat_num_minpack):
    accepted_before = count_solved
    nfev_before = total_nfev
    r = None
    # [modify-lunar] Use the shared augmented initial guess: [lambda(0), tf, q0].
    x0 = augmented_initial_guess_pool[i].copy()  # [modify-lunar]

    time_start = time.perf_counter()

    r = root(
        shooting_augmented_gate,
        x0,
        args=(logeps0, logepsf, rho0, delta_rho, gate_p, gate_option, lambda0, False, None),
        method='hybr',
        tol=minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            # [modify-lunar] 8 base unknowns plus q.
            'diag': minpack_diag + [1.0],
            'eps': fd_eps,
        }
    )

    time_end = time.perf_counter()
    total_time += time_end - time_start
    total_nfev += r['nfev']

    res = np.linalg.norm(r['fun'])
    if r['success'] and res < minpack_error:
        count_solved += 1
        converge_nfev += r['nfev']

        print(f"counter: {count_solved}/{i} nfev: {r['nfev']}", end='', flush=True)

        shooting_augmented_gate(
            r['x'],
            logeps0, logepsf, rho0, delta_rho, gate_p, gate_option, lambda0,
            out=True,
            Time_out=None,
        )

        final_mass = ta.state[6] * MUnit
        if final_mass > global_mf:
            global_mf = final_mass
            global_lambda = r['x']
    save_trial(i, r, count_solved > accepted_before, total_nfev - nfev_before)


# =========================
# Output convergence rate and average convergence time
# =========================
if count_solved < 1:
    print("Fail to solve the problem!")
else:
    print("Succeed to solve the problem!")
    print(f"Solved: {count_solved} in {repeat_num_minpack} ({count_solved/repeat_num_minpack:.4f}), terminal mass: {global_mf:.4f}kg")
    print(f"Average number of function calls for Minpack: {total_nfev/repeat_num_minpack:.0f}")
    print(f"Average number of function calls for convergent solution: {converge_nfev/count_solved:.0f}")
    # [modify-lunar] global_lambda contains [lambda(0), tf, q].
    print("global_lambda_tf_q:", global_lambda)
    print(f"global tf: {global_lambda[7] * TUnit:.12f} s")
records_by_method['Gated_Rational_p1'] = records


Succeed to solve the problem!
Solved: 1229 in 5000 (0.2458), terminal mass: 510.4080kg
Average number of function calls for Minpack: 63
Average number of function calls for convergent solution: 149
global_lambda_tf_q: [0.9470997368460344, -0.15229396603930498, 0.28755556708659674, 0.20158655395774674, 0.0880991977199855, 0.3201581537770192, 0.45531025841321854, 0.6695497278716367, 0.9999994593233389]
global tf: 692.843907069579 s


In [46]:
# =========================
# Computational time: all / successful / failed attempts
# =========================
timing_ids = np.arange(500)
accepted = np.array([r['accepted'] for r in records_by_method['Gated_Rational_p1']], dtype=bool)
successful_ids = timing_ids[accepted[timing_ids]]
failed_ids = timing_ids[~accepted[timing_ids]]
cpu_samples.clear()


In [47]:
%%timeit -n 1 -r 7 -v timing_all prepare_timing()
for i in timing_ids:
    sample_cpu()
    # [modify-lunar] Use the shared augmented initial guess: [lambda(0), tf, q0].
    x0 = augmented_initial_guess_pool[i].copy()  # [modify-lunar]


    r = root(
        shooting_augmented_gate,
        x0,
        args=(logeps0, logepsf, rho0, delta_rho, gate_p, gate_option, lambda0, False, None),
        method='hybr',
        tol=minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            # [modify-lunar] 8 base unknowns plus q.
            'diag': minpack_diag + [1.0],
            'eps': fd_eps,
        }
    )


1.67 s ± 20 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [48]:
%%timeit -n 1 -r 7 -v timing_successful prepare_timing()
for i in successful_ids:
    sample_cpu()
    # [modify-lunar] Use the shared augmented initial guess: [lambda(0), tf, q0].
    x0 = augmented_initial_guess_pool[i].copy()  # [modify-lunar]


    r = root(
        shooting_augmented_gate,
        x0,
        args=(logeps0, logepsf, rho0, delta_rho, gate_p, gate_option, lambda0, False, None),
        method='hybr',
        tol=minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            # [modify-lunar] 8 base unknowns plus q.
            'diag': minpack_diag + [1.0],
            'eps': fd_eps,
        }
    )


684 ms ± 14.1 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [49]:
%%timeit -n 1 -r 7 -v timing_failed prepare_timing()
for i in failed_ids:
    sample_cpu()
    # [modify-lunar] Use the shared augmented initial guess: [lambda(0), tf, q0].
    x0 = augmented_initial_guess_pool[i].copy()  # [modify-lunar]


    r = root(
        shooting_augmented_gate,
        x0,
        args=(logeps0, logepsf, rho0, delta_rho, gate_p, gate_option, lambda0, False, None),
        method='hybr',
        tol=minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            # [modify-lunar] 8 base unknowns plus q.
            'diag': minpack_diag + [1.0],
            'eps': fd_eps,
        }
    )


990 ms ± 16.9 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [50]:
all_ms = timing_all.average * 1000.0 / len(timing_ids)
p_success = accepted.mean()
results_by_method['Gated_Rational_p1'] = dict(
    method='Option 2: Rational p=1, A=10', trials=len(accepted), accepted=int(accepted.sum()),
    first_ms=all_ms / p_success if p_success else float('inf'),
    successful_ms=timing_successful.average * 1000.0 / len(successful_ids) if len(successful_ids) else float('nan'),
    failed_ms=timing_failed.average * 1000.0 / len(failed_ids) if len(failed_ids) else float('nan'),
    all_ms=all_ms,
    nfev_successful=float(np.mean([r['nfev'] for r in records_by_method['Gated_Rational_p1'] if r['accepted']])) if accepted.any() else float('nan'),
    nfev_all=float(np.mean([r['nfev'] for r in records_by_method['Gated_Rational_p1']])),
    timing_trials=len(timing_ids), timing_successful_trials=len(successful_ids), timing_failed_trials=len(failed_ids),
    timeit_runs_seconds=timing_all.all_runs,
    timeit_successful_seconds=timing_successful.all_runs,
    timeit_failed_seconds=timing_failed.all_runs,
    cpu_samples=dict(Counter(cpu_samples)),
    fast_core_sample_pct=100.0 * sum(cpu in fast_cpus for cpu in cpu_samples) / len(cpu_samples) if cpu_samples else None)


## Option 3: Rational p=2, A=10


In [51]:
# =========================
# Initialize solving time and global optimal solution.
# =========================
eps0 = 1.0e-0
epsf = 1.0e-5
logeps0 = np.log10(eps0)
logepsf = np.log10(epsf)

# [MODIFIED] parameters for the augmented residual
gate_option = 'rational'
gate_p = 2.0
rho0 = 0.029317332

# [other parameters]:
fd_eps = 0.
delta_rho = 1.0e-8


# [general settings]:
lambda0 = 1.0

count_solved = 0
total_time = 0.0
total_nfev = 0
converge_nfev = 0

global_lambda = None
global_mf = -1.0

# =========================
# Shared initial guess loop
# =========================


In [52]:
records = []
for i in range(repeat_num_minpack):
    accepted_before = count_solved
    nfev_before = total_nfev
    r = None
    # [modify-lunar] Use the shared augmented initial guess: [lambda(0), tf, q0].
    x0 = augmented_initial_guess_pool[i].copy()  # [modify-lunar]

    time_start = time.perf_counter()

    r = root(
        shooting_augmented_gate,
        x0,
        args=(logeps0, logepsf, rho0, delta_rho, gate_p, gate_option, lambda0, False, None),
        method='hybr',
        tol=minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            # [modify-lunar] 8 base unknowns plus q.
            'diag': minpack_diag + [1.0],
            'eps': fd_eps,
        }
    )

    time_end = time.perf_counter()
    total_time += time_end - time_start
    total_nfev += r['nfev']

    res = np.linalg.norm(r['fun'])
    if r['success'] and res < minpack_error:
        count_solved += 1
        converge_nfev += r['nfev']

        print(f"counter: {count_solved}/{i} nfev: {r['nfev']}", end='', flush=True)

        shooting_augmented_gate(
            r['x'],
            logeps0, logepsf, rho0, delta_rho, gate_p, gate_option, lambda0,
            out=True,
            Time_out=None,
        )

        final_mass = ta.state[6] * MUnit
        if final_mass > global_mf:
            global_mf = final_mass
            global_lambda = r['x']
    save_trial(i, r, count_solved > accepted_before, total_nfev - nfev_before)


# =========================
# Output convergence rate and average convergence time
# =========================
if count_solved < 1:
    print("Fail to solve the problem!")
else:
    print("Succeed to solve the problem!")
    print(f"Solved: {count_solved} in {repeat_num_minpack} ({count_solved/repeat_num_minpack:.4f}), terminal mass: {global_mf:.4f}kg")
    print(f"Average number of function calls for Minpack: {total_nfev/repeat_num_minpack:.0f}")
    print(f"Average number of function calls for convergent solution: {converge_nfev/count_solved:.0f}")
    # [modify-lunar] global_lambda contains [lambda(0), tf, q].
    print("global_lambda_tf_q:", global_lambda)
    print(f"global tf: {global_lambda[7] * TUnit:.12f} s")
records_by_method['Gated_Rational_p2'] = records


Succeed to solve the problem!
Solved: 1238 in 5000 (0.2476), terminal mass: 510.4080kg
Average number of function calls for Minpack: 62
Average number of function calls for convergent solution: 142
global_lambda_tf_q: [0.9470997361352349, -0.1522939657561967, 0.2875555671205403, 0.20158655373293075, 0.0880991978422181, 0.32015815384763635, 0.45531025835310995, 0.6695497279498381, 1.0000000000200824]
global tf: 692.843907150502 s


In [53]:
# =========================
# Computational time: all / successful / failed attempts
# =========================
timing_ids = np.arange(500)
accepted = np.array([r['accepted'] for r in records_by_method['Gated_Rational_p2']], dtype=bool)
successful_ids = timing_ids[accepted[timing_ids]]
failed_ids = timing_ids[~accepted[timing_ids]]
cpu_samples.clear()


In [54]:
%%timeit -n 1 -r 7 -v timing_all prepare_timing()
for i in timing_ids:
    sample_cpu()
    # [modify-lunar] Use the shared augmented initial guess: [lambda(0), tf, q0].
    x0 = augmented_initial_guess_pool[i].copy()  # [modify-lunar]


    r = root(
        shooting_augmented_gate,
        x0,
        args=(logeps0, logepsf, rho0, delta_rho, gate_p, gate_option, lambda0, False, None),
        method='hybr',
        tol=minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            # [modify-lunar] 8 base unknowns plus q.
            'diag': minpack_diag + [1.0],
            'eps': fd_eps,
        }
    )


1.56 s ± 14.5 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [55]:
%%timeit -n 1 -r 7 -v timing_successful prepare_timing()
for i in successful_ids:
    sample_cpu()
    # [modify-lunar] Use the shared augmented initial guess: [lambda(0), tf, q0].
    x0 = augmented_initial_guess_pool[i].copy()  # [modify-lunar]


    r = root(
        shooting_augmented_gate,
        x0,
        args=(logeps0, logepsf, rho0, delta_rho, gate_p, gate_option, lambda0, False, None),
        method='hybr',
        tol=minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            # [modify-lunar] 8 base unknowns plus q.
            'diag': minpack_diag + [1.0],
            'eps': fd_eps,
        }
    )


593 ms ± 1.16 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [56]:
%%timeit -n 1 -r 7 -v timing_failed prepare_timing()
for i in failed_ids:
    sample_cpu()
    # [modify-lunar] Use the shared augmented initial guess: [lambda(0), tf, q0].
    x0 = augmented_initial_guess_pool[i].copy()  # [modify-lunar]


    r = root(
        shooting_augmented_gate,
        x0,
        args=(logeps0, logepsf, rho0, delta_rho, gate_p, gate_option, lambda0, False, None),
        method='hybr',
        tol=minpack_tol,
        options={
            'xtol': minpack_xtol,
            'maxfev': minpack_maxfev,
            'factor': minpack_factor,
            # [modify-lunar] 8 base unknowns plus q.
            'diag': minpack_diag + [1.0],
            'eps': fd_eps,
        }
    )


955 ms ± 4.17 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [57]:
all_ms = timing_all.average * 1000.0 / len(timing_ids)
p_success = accepted.mean()
results_by_method['Gated_Rational_p2'] = dict(
    method='Option 3: Rational p=2, A=10', trials=len(accepted), accepted=int(accepted.sum()),
    first_ms=all_ms / p_success if p_success else float('inf'),
    successful_ms=timing_successful.average * 1000.0 / len(successful_ids) if len(successful_ids) else float('nan'),
    failed_ms=timing_failed.average * 1000.0 / len(failed_ids) if len(failed_ids) else float('nan'),
    all_ms=all_ms,
    nfev_successful=float(np.mean([r['nfev'] for r in records_by_method['Gated_Rational_p2'] if r['accepted']])) if accepted.any() else float('nan'),
    nfev_all=float(np.mean([r['nfev'] for r in records_by_method['Gated_Rational_p2']])),
    timing_trials=len(timing_ids), timing_successful_trials=len(successful_ids), timing_failed_trials=len(failed_ids),
    timeit_runs_seconds=timing_all.all_runs,
    timeit_successful_seconds=timing_successful.all_runs,
    timeit_failed_seconds=timing_failed.all_runs,
    cpu_samples=dict(Counter(cpu_samples)),
    fast_core_sample_pct=100.0 * sum(cpu in fast_cpus for cpu in cpu_samples) / len(cpu_samples) if cpu_samples else None)


In [58]:

headings = ['Method', 'Success / 5000', 'Convergence [%]', 'Expected first [ms]',
            'Successful mean [ms]', 'Failed mean [ms]', 'All-trial mean [ms]']
table = ['| '+' | '.join(headings)+' |', '| '+' | '.join(['---']*len(headings))+' |']
for value in results_by_method.values():
    values = [value['method'], value['accepted'], f"{100.*value['accepted']/value['trials']:.2f}"]
    values += [f"{value[k]:.3f}" for k in ('first_ms','successful_ms','failed_ms','all_ms')]
    table.append('| '+' | '.join(map(str,values))+' |')
display(Markdown('\n'.join(table)))

| Method | Success / 5000 | Convergence [%] | Expected first [ms] | Successful mean [ms] | Failed mean [ms] | All-trial mean [ms] |
| --- | --- | --- | --- | --- | --- | --- |
| Single shooting, epsilon=1 | 1170 | 23.40 | 8.913 | 1.361 | 2.289 | 2.086 |
| Single shooting, epsilon=1e-5 | 788 | 15.76 | 17.042 | 4.690 | 2.443 | 2.686 |
| Stepwise | 1170 | 23.40 | 15.026 | 7.764 | 2.391 | 3.516 |
| Ungated | 948 | 18.96 | 16.060 | 5.425 | 2.522 | 3.045 |
| Option 1: Exp p=1, A=10 | 1245 | 24.90 | 12.662 | 5.093 | 2.530 | 3.153 |
| Option 2: Rational p=1, A=10 | 1229 | 24.58 | 13.583 | 6.054 | 2.558 | 3.339 |
| Option 3: Rational p=2, A=10 | 1238 | 24.76 | 12.615 | 5.071 | 2.494 | 3.123 |

15004765